# E9: Feature Selection Funnel — L5 (Multivariate Stability Selection, Model A)

**Spec sections**: 6.6, D.6, 6.8.  
**Input**: DEV GOOD+BAD; L4 pool (`feature_lists/l4_pool.json`); WoE from the fixed L1 bins.  
**Output**: `feature_lists/model_a.json`, feature-log L5 + summary columns, `reports/03_L5.xlsx`, `reports/03_funnel_summary.xlsx`, `reports/run_summaries/s03_L5.json`.

Note (6.6.1): C* is chosen with the **fixed L1 bins**; binning is not refitted per fold.

In [0]:
%pip install "optbinning" "numpy<2" "pyarrow==14.0.1" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, hashlib, shutil, tempfile, json, time, pickle, warnings, importlib
from datetime import datetime

def _find_root(start):
    """Walk up from the notebook folder until config/config.yaml is found."""
    p = os.path.abspath(start)
    while p != "/":
        if os.path.exists(os.path.join(p, "config", "config.yaml")):
            return p
        p = os.path.dirname(p)
    raise FileNotFoundError(f"config/config.yaml not found above {start}")

PROJECT_ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import common
importlib.reload(common)

warnings.filterwarnings("ignore", category=FutureWarning)

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS   = common.artifacts_path(cfg)
REPORTS     = os.path.join(ARTIFACTS, "reports")
RUN_SUMS    = os.path.join(REPORTS, "run_summaries")
BINNING_DIR = os.path.join(ARTIFACTS, "binning", "L1")
FEAT_LISTS  = os.path.join(ARTIFACTS, "feature_lists")
for d in (REPORTS, RUN_SUMS, FEAT_LISTS):
    os.makedirs(d, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"

L5 = cfg["funnel"]["L5"]
assert L5["c_rule"] == "one_se", f"Unsupported c_rule {L5['c_rule']} (spec 6.6.1 defines one_se only)"
C_GRID      = np.logspace(L5["c_grid"][0], L5["c_grid"][1], int(L5["c_grid"][2]))
PVALUE_MAX  = L5["pvalue_max"]
PTS_FLAG    = L5["points_share_flag"]
N_LO, N_HI  = L5["n_features"]
VIF_MAX     = cfg["funnel"]["L4"]["vif_max"]
N_SPLITS    = cfg["cv"]["r_folds"]["n_splits"]
FACTOR      = cfg["scaling"]["pdo"] / np.log(2)

target_col = cfg["columns"]["target"]
status_col = cfg["columns"]["status"]

# scikit-learn parameter form for L1 logistic regression
SK_VER = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
def make_l1_lr(C):
    if SK_VER >= (1, 8):   # 1.8+: penalty deprecated, use l1_ratio form
        return LogisticRegression(l1_ratio=1.0, solver="liblinear", C=C, max_iter=2000)
    return LogisticRegression(penalty="l1", solver="liblinear", C=C, max_iter=2000)
assert SK_VER < (1, 8), ("scikit-learn >= 1.8: common.stability_selection must be updated to the l1_ratio form "
                         "before running E9")

# Git commit of the project folder
def git_commit(root):
    try:
        head = open(os.path.join(root, ".git", "HEAD")).read().strip()
        if not head.startswith("ref:"):
            return head
        ref = head.split(" ", 1)[1]
        p = os.path.join(root, ".git", ref)
        if os.path.exists(p):
            return open(p).read().strip()
        pr = os.path.join(root, ".git", "packed-refs")
        if os.path.exists(pr):
            for line in open(pr):
                if line.strip().endswith(ref):
                    return line.split()[0]
    except Exception:
        pass
    return "unknown"
GIT_COMMIT = git_commit(PROJECT_ROOT)

# Funnel config hash check (frozen at E4)
funnel_config_hash = hashlib.sha256(json.dumps(cfg["funnel"], sort_keys=True).encode()).hexdigest()
with open(os.path.join(RUN_SUMS, "s03_L0.json")) as f:
    l0_funnel_hash = json.load(f)["funnel_config_hash"]
tc_path = os.path.join(PROJECT_ROOT, "config", "threshold_changes.csv")
if funnel_config_hash != l0_funnel_hash:
    tc = pd.read_csv(tc_path) if os.path.exists(tc_path) else pd.DataFrame()
    if len(tc) == 0:
        raise ValueError(f"Funnel config hash mismatch! L0 frozen {l0_funnel_hash[:12]} vs current "
                         f"{funnel_config_hash[:12]} and no row in threshold_changes.csv")
    print(f"\u26a0 Funnel config hash changed; threshold_changes.csv has {len(tc)} row(s) \u2014 proceeding.")
else:
    print(f"\u2705 Funnel config hash matches L0: {funnel_config_hash[:12]}")

_TMP = tempfile.mkdtemp(prefix="e9_")
_t0 = time.time()
print(f"Project root : {PROJECT_ROOT}")
print(f"Git commit   : {GIT_COMMIT[:12]}")
print(f"Config hash  : {common.config_hash(cfg)[:12]}")
print(f"Versions     : sklearn {sklearn.__version__}, statsmodels {statsmodels.__version__}")
print(f"L5 params    : C grid {C_GRID[0]:.4g}..{C_GRID[-1]:.4g} ({len(C_GRID)}), n_sub={L5['n_subsamples']}, "
      f"frac={L5['subsample_frac']}, freq={L5['select_freq']}, floor={L5['select_freq_floor']}, "
      f"n_features={L5['n_features']}, p<{PVALUE_MAX}, VIF<={VIF_MAX}, points_share_flag={PTS_FLAG}")

In [0]:
# ── DEV only (OOT never loaded): Spark filter, then common.load_window ──
master_dev = spark.table(MASTER_TBL).filter("window = 'DEV'").toPandas()
dev = common.load_window(master_dev, "DEV", cfg)
dev_gb = dev[dev[status_col].isin(cfg["labelled_statuses"])].reset_index(drop=True)
common.assert_labelled(dev_gb, cfg)
y = (dev_gb[status_col] == "BAD").astype(int)
assert (dev_gb[target_col].astype(int).isin([0, 1])).all()
assert ((dev_gb[target_col].astype(int) == 1) == (y == 1)).all(), "sample_status / target mismatch"

exp = cfg["expected_counts"]["DEV"]
print(f"DEV GOOD+BAD: {len(dev_gb):,}  Bads: {int(y.sum())}  (expected {exp['records']:,} / {exp['bads']})")
assert len(dev_gb) == exp["records"] and int(y.sum()) == exp["bads"], "DEV counts differ from config.expected_counts"

# Stratification key: target + "_" + app_quarter (spec 5.5.1)
strat = dev_gb[target_col].astype(int).astype(str) + "_" + dev_gb["app_quarter"].astype(str)
print(f"Strat key levels: {strat.nunique()}  (min count {strat.value_counts().min()})")

# L4 pool
with open(os.path.join(FEAT_LISTS, "l4_pool.json")) as f:
    l4_pool = json.load(f)
pool_feats = [r["feature"] for r in l4_pool]

flog = spark.table(FLOG_TBL).toPandas()
l4_y = set(flog.loc[flog["L4_pass"] == "Y", "feature"])
assert set(pool_feats) == l4_y, "l4_pool.json differs from feature-log L4_pass = Y"
print(f"L4 pool: {len(pool_feats)} features  | feature log rows: {len(flog)}")

# Fixed L1 binning objects and WoE matrix (same transform as L4)
binning_objects = {}
for feat in pool_feats:
    with open(os.path.join(BINNING_DIR, f"{feat}.pkl"), "rb") as f:
        binning_objects[feat] = pickle.load(f)

def safe_woe(ob, X):
    try:
        return ob.transform(X, metric="woe")
    except TypeError:
        return ob.transform(pd.to_numeric(X, errors="coerce"), metric="woe")

X_woe = pd.DataFrame({f: safe_woe(binning_objects[f], dev_gb[f]) for f in pool_feats}, index=dev_gb.index)
X_woe = X_woe.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)
print(f"WoE matrix: {X_woe.shape}, NaN after fill: {int(X_woe.isna().sum().sum())}")

fdict = pd.read_csv(os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")).set_index("feature")

In [0]:
# ── 6.6.1 C* via 5-fold stratified CV, L1-LR, 1-SE rule ──
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X_woe, strat))

rows = []
for C in C_GRID:
    aucs = []
    for tr, va in folds:
        lr = make_l1_lr(C).fit(X_woe.iloc[tr], y.iloc[tr])
        aucs.append(roc_auc_score(y.iloc[va], lr.predict_proba(X_woe.iloc[va])[:, 1]))
    n_nz = int((np.abs(make_l1_lr(C).fit(X_woe, y).coef_[0]) > 1e-8).sum())
    sd = float(np.std(aucs, ddof=1))
    rows.append({"C": C, "mean_auc": float(np.mean(aucs)), "std_auc": sd,
                 "se_auc": sd / np.sqrt(N_SPLITS), "n_nonzero_full_dev": n_nz,
                 **{f"auc_fold{k+1}": a for k, a in enumerate(aucs)}})
c_path = pd.DataFrame(rows)

best_i = int(c_path["mean_auc"].idxmax())
BEST_C, BEST_AUC, BEST_SE = c_path.loc[best_i, ["C", "mean_auc", "se_auc"]]
AUC_THR = BEST_AUC - BEST_SE
C_STAR = float(c_path.loc[c_path["mean_auc"] >= AUC_THR, "C"].min())
c_path["is_best"] = c_path.index == best_i
c_path["is_C_star"] = np.isclose(c_path["C"], C_STAR)
CSTAR_ROW = c_path[c_path["is_C_star"]].iloc[0]

print(f"Best C = {BEST_C:.4g}: mean AUC {BEST_AUC:.4f}, SE {BEST_SE:.4f}  (SE = std(ddof=1)/sqrt({N_SPLITS}))")
print(f"1-SE threshold = {AUC_THR:.4f}")
print(f"C* = {C_STAR:.4g}: mean AUC {CSTAR_ROW['mean_auc']:.4f}, non-zero coefs on DEV {CSTAR_ROW['n_nonzero_full_dev']}")
display(c_path[["C", "mean_auc", "std_auc", "se_auc", "n_nonzero_full_dev", "is_best", "is_C_star"]])

In [0]:
# ── 6.6.2 Stability selection with fixed C* (seed + i, stratified on key) ──
freq = common.stability_selection(X_woe[pool_feats], y, strat, C_STAR, cfg)
sel_feats, THR_USED = common.pick_features(freq, cfg)

print(f"Subsamples: {L5['n_subsamples']} x {L5['subsample_frac']:.0%}, C* = {C_STAR:.4g}")
for t in L5["select_freq"] + [L5["select_freq_floor"]]:
    print(f"  freq >= {t:.2f}: {int((freq >= t).sum())} features")
print(f"Threshold used: {THR_USED:.2f}  -> {len(sel_feats)} features selected")

freq_df = freq.rename("select_freq").reset_index().rename(columns={"index": "feature"})
freq_df["selected"] = freq_df["feature"].isin(sel_feats).map({True: "Y", False: "N"})
freq_df["source"] = freq_df["feature"].map(fdict["source"]) if "source" in fdict.columns else ""
display(freq_df)

In [0]:
# ── 6.6.3 Unregularised statsmodels Logit on WoE (with constant) ──
def fit_logit(feats):
    X = sm.add_constant(X_woe[feats], has_constant="add")
    return sm.Logit(y, X).fit(disp=0, maxiter=200)

# Reference sign for a tie in sign counts: sign shared by univariate WoE fits
uni_sign = pd.Series({f: np.sign(fit_logit([f]).params[f]) for f in sel_feats})
REF_SIGN = 1.0 if (uni_sign > 0).sum() >= (uni_sign < 0).sum() else -1.0
print(f"Univariate WoE coefficient signs: {uni_sign.value_counts().to_dict()}  -> reference sign {REF_SIGN:+.0f}")

cur = list(sel_feats)
logit_steps, l5_drop = [], {}   # l5_drop: feature -> (rule, coef, p)
for step in range(1, len(sel_feats) + 1):
    res = fit_logit(cur)
    coefs, pv = res.params.drop("const"), res.pvalues.drop("const")
    signs = np.sign(coefs)
    n_pos, n_neg = int((signs > 0).sum()), int((signs < 0).sum())
    if n_pos and n_neg:
        minority = (1.0 if n_pos < n_neg else -1.0) if n_pos != n_neg else -REF_SIGN
        f = pv[signs == minority].idxmax()
        rule = "LOGIT_SIGN"
    elif (pv >= PVALUE_MAX).any():
        f = pv.idxmax()
        rule = "LOGIT_PVALUE"
    else:
        logit_steps.append({"step": step, "n_features": len(cur), "action": "STABLE", "feature": None,
                            "coef": None, "p_value": None, "n_pos": n_pos, "n_neg": n_neg,
                            "converged": res.mle_retvals["converged"]})
        break
    l5_drop[f] = (rule, float(coefs[f]), float(pv[f]))
    logit_steps.append({"step": step, "n_features": len(cur), "action": rule, "feature": f,
                        "coef": float(coefs[f]), "p_value": float(pv[f]), "n_pos": n_pos, "n_neg": n_neg,
                        "converged": res.mle_retvals["converged"]})
    cur.remove(f)
    print(f"  step {step}: drop {f} [{rule}] coef={coefs[f]:+.4f} p={pv[f]:.4f}")

final_feats = list(cur)
logit_final = fit_logit(final_feats)
logit_steps = pd.DataFrame(logit_steps)

# VIF check on the final set (no automatic drop)
def compute_vif(features):
    X = np.column_stack([np.ones(len(X_woe)), X_woe[features].values])
    return {f: float(variance_inflation_factor(X, i + 1)) for i, f in enumerate(features)}
final_vif = compute_vif(final_feats) if len(final_feats) > 1 else {final_feats[0]: 1.0}

logit_tbl = pd.DataFrame({"coef": logit_final.params, "std_err": logit_final.bse,
                          "z": logit_final.tvalues, "p_value": logit_final.pvalues})
logit_tbl["vif"] = logit_tbl.index.map(final_vif)
print(f"\nFinal Logit: {len(final_feats)} features, converged={logit_final.mle_retvals['converged']}, "
      f"pseudo R2={logit_final.prsquared:.4f}, AUC(DEV, in-sample)={roc_auc_score(y, logit_final.predict()):.4f}")
print(f"Signs: {np.sign(logit_final.params.drop('const')).value_counts().to_dict()}  "
      f"max p={logit_final.pvalues.drop('const').max():.4g}  max VIF={max(final_vif.values()):.2f}")
display(logit_tbl.reset_index().rename(columns={"index": "term"}))

In [0]:
# ── Points concentration (flag only, no drop) ──
# points per bin = -coef * WoE * Factor, Factor = pdo / ln(2); populated bins of the fixed L1 binning table
pts_rows = []
for f in final_feats:
    bt = binning_objects[f].binning_table.build()
    bt = bt[bt.index != "Totals"].copy()
    bt["Count"] = pd.to_numeric(bt["Count"], errors="coerce")
    bt["WoE"] = pd.to_numeric(bt["WoE"], errors="coerce")
    bt = bt[bt["Count"] > 0]
    pts = -logit_final.params[f] * bt["WoE"] * FACTOR
    pts_rows.append({"feature": f, "coef": float(logit_final.params[f]), "n_bins_populated": len(bt),
                     "woe_min": float(bt["WoE"].min()), "woe_max": float(bt["WoE"].max()),
                     "points_min": float(pts.min()), "points_max": float(pts.max()),
                     "points_range": float(pts.max() - pts.min())})
points_df = pd.DataFrame(pts_rows)
points_df["points_range_share"] = points_df["points_range"] / points_df["points_range"].sum()
points_df["points_review_flag"] = np.where(points_df["points_range_share"] > PTS_FLAG, "Y", "N")
points_df = points_df.sort_values("points_range_share", ascending=False).reset_index(drop=True)

print(f"Factor = {FACTOR:.4f}; total points range = {points_df['points_range'].sum():.1f}")
flagged = points_df.loc[points_df["points_review_flag"] == "Y", "feature"].tolist()
print(f"Features above points share {PTS_FLAG:.0%}: {flagged if flagged else 'none'}")
display(points_df)

In [0]:
# ── feature_lists/model_a.json ──
pts_idx = points_df.set_index("feature")
model_a = {
    "module": "E9_L5", "run_time": datetime.now().isoformat(), "git_commit": GIT_COMMIT,
    "config_hash": common.config_hash(cfg), "funnel_config_hash": funnel_config_hash,
    "C_star": C_STAR, "select_freq_threshold": THR_USED,
    "intercept": float(logit_final.params["const"]), "n_features": len(final_feats),
    "features": [{
        "feature": f,
        "source": fdict.loc[f, "source"] if f in fdict.index and "source" in fdict.columns else "",
        "dtype": fdict.loc[f, "dtype"] if f in fdict.index and "dtype" in fdict.columns else "",
        "select_freq": float(freq[f]), "coef": float(logit_final.params[f]),
        "p_value": float(logit_final.pvalues[f]), "vif": final_vif[f],
        "points_range": float(pts_idx.loc[f, "points_range"]),
        "points_range_share": float(pts_idx.loc[f, "points_range_share"]),
        "points_review_flag": pts_idx.loc[f, "points_review_flag"],
    } for f in final_feats],
}
model_a_path = os.path.join(FEAT_LISTS, "model_a.json")
with open(model_a_path, "w") as fh:
    json.dump(model_a, fh, indent=2, default=str)
print(f"Saved {model_a_path} ({len(final_feats)} features)")

# ── Feature-log L5 + summary columns (idempotent) ──
L5_COLS = ["select_freq", "coef", "p_value", "points_range_share", "points_review_flag",
           "L5_pass", "L5_reason", "final_A"]
fl = flog.drop(columns=[c for c in L5_COLS if c in flog.columns]).copy()

def _is_missing(s):
    return s.isna() | s.astype(str).str.strip().isin(["", "nan", "None", "<NA>", "NaN"])

# Reset own previous L5 drops
for c in ("dropped_at", "drop_rule"):
    fl[c] = fl[c].where(~_is_missing(fl[c]), None).astype(object)
prev_l5 = fl["dropped_at"] == "L5"
fl.loc[prev_l5, ["dropped_at", "drop_rule"]] = None

recs = []
for f in pool_feats:
    r = {"feature": f, "select_freq": float(freq[f])}
    if f in final_feats:
        r.update(coef=float(logit_final.params[f]), p_value=float(logit_final.pvalues[f]),
                 points_range_share=float(pts_idx.loc[f, "points_range_share"]),
                 points_review_flag=pts_idx.loc[f, "points_review_flag"], L5_pass="Y", L5_reason=None)
    elif f in l5_drop:
        rule, cf, p = l5_drop[f]
        r.update(coef=cf, p_value=p, points_range_share=np.nan, points_review_flag=None,
                 L5_pass="N", L5_reason=f"{rule}:p={p:.4f}")
    else:
        r.update(coef=np.nan, p_value=np.nan, points_range_share=np.nan, points_review_flag=None,
                 L5_pass="N", L5_reason=f"SELECT_FREQ<{THR_USED:.2f}:{freq[f]:.2f}")
    recs.append(r)
l5_df = pd.DataFrame(recs)
fl = fl.merge(l5_df, on="feature", how="left")

fl["final_A"] = np.where(fl["feature"].isin(final_feats), "Y", "N")
fl["in_pool_BC"] = np.where(fl["feature"].isin(pool_feats), "Y", "N")
m = (fl["L5_pass"] == "N") & fl["dropped_at"].isna()
fl.loc[m, "dropped_at"] = "L5"
fl.loc[m, "drop_rule"] = fl.loc[m, "L5_reason"]
assert len(fl) == len(flog) and fl["feature"].is_unique, "feature-log row count changed"

# Spark-safe types
for c in fl.columns:
    if fl[c].dtype == object:
        fl[c] = fl[c].where(fl[c].notna(), None)
    if fl[c].isna().all():
        fl[c] = fl[c].astype(str)

spark.createDataFrame(fl).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FLOG_TBL)
print(f"Saved table: {FLOG_TBL} ({spark.table(FLOG_TBL).count()} rows)")

csv_paths = [os.path.join(ARTIFACTS, "feature_log.csv"), os.path.join(REPORTS, "feature_log.csv")]
for p in csv_paths:
    fl.to_csv(p, index=False)
    print(f"Saved CSV  : {p}")
flog_updated = fl
print(fl["dropped_at"].value_counts(dropna=False).to_dict(), "| final_A:", fl["final_A"].value_counts().to_dict())

In [0]:
# ── reports/03_L5.xlsx and reports/03_funnel_summary.xlsx ──
BINNING_NOTE = ("C* chosen with 5-fold stratified CV on the fixed L1 bins; binning was NOT refitted per fold "
                "(allowed by spec 6.6.1).")
few_explanation = None
if len(final_feats) < N_LO:
    few_explanation = (f"E9/L5 ({datetime.now():%Y-%m-%d}): only {len(final_feats)} final features (< {N_LO}) after "
                       f"the select_freq floor {L5['select_freq_floor']} and Logit checks. Selected at threshold "
                       f"{THR_USED:.2f}: {len(sel_feats)}; dropped by Logit checks: {list(l5_drop)}. Spec 6.6.2 keeps "
                       f"the actual result; model owner to confirm acceptance of Model A with this feature count.")
    with open(os.path.join(REPORTS, "open_questions.md"), "a") as fh:
        fh.write(f"\n- {few_explanation}\n")
    print("Written to open_questions.md:", few_explanation)

def _meta(module):
    return pd.DataFrame([{"module": module, "run_time": datetime.now().isoformat(), "git_commit": GIT_COMMIT,
                          "config_sha256": common.config_hash(cfg), "funnel_config_hash": funnel_config_hash}])

final_list = pd.DataFrame(model_a["features"])
selection_info = pd.DataFrame([
    {"item": "C_grid", "value": f"logspace({L5['c_grid'][0]}, {L5['c_grid'][1]}, {L5['c_grid'][2]})"},
    {"item": "best_C", "value": BEST_C}, {"item": "best_mean_auc", "value": BEST_AUC},
    {"item": "best_se_auc", "value": BEST_SE}, {"item": "one_se_threshold", "value": AUC_THR},
    {"item": "C_star", "value": C_STAR}, {"item": "C_star_mean_auc", "value": CSTAR_ROW["mean_auc"]},
    {"item": "binning_note", "value": BINNING_NOTE},
    {"item": "n_subsamples", "value": L5["n_subsamples"]}, {"item": "subsample_frac", "value": L5["subsample_frac"]},
    {"item": "select_freq_threshold_used", "value": THR_USED},
    {"item": "n_selected_at_threshold", "value": len(sel_feats)},
    {"item": "n_final", "value": len(final_feats)},
    {"item": "reference_sign", "value": REF_SIGN},
    {"item": "few_features_explanation", "value": few_explanation or "n/a"},
])

tmp = os.path.join(_TMP, "03_L5.xlsx")
with pd.ExcelWriter(tmp, engine="openpyxl") as w:
    m = _meta("E9_L5"); m["binning_note"] = BINNING_NOTE
    m.to_excel(w, sheet_name="meta", index=False)
    selection_info.to_excel(w, sheet_name="selection_info", index=False)
    c_path.to_excel(w, sheet_name="C_path", index=False)
    freq_df.to_excel(w, sheet_name="select_freq", index=False)
    logit_steps.to_excel(w, sheet_name="logit_steps", index=False)
    logit_tbl.reset_index().rename(columns={"index": "term"}).to_excel(w, sheet_name="logit_summary", index=False)
    pd.DataFrame({"summary": str(logit_final.summary()).split("\n")}).to_excel(w, sheet_name="logit_summary_text", index=False)
    points_df.to_excel(w, sheet_name="points_concentration", index=False)
    final_list.to_excel(w, sheet_name="final_list", index=False)
shutil.copyfile(tmp, os.path.join(REPORTS, "03_L5.xlsx"))
print(f"Saved {REPORTS}/03_L5.xlsx")

# Funnel summary: features in / dropped / out per layer and per source
LAYERS = ["L0", "L1", "L2", "L3", "L4", "L5"]
src_col = fl["source"].astype(str) if "source" in fl.columns else pd.Series("", index=fl.index)
def funnel_rows(mask, label):
    out, n_in = [], int(mask.sum())
    for L in LAYERS:
        n_drop = int((mask & (fl["dropped_at"] == L)).sum())
        out.append({"source": label, "layer": L, "features_in": n_in, "dropped": n_drop, "features_out": n_in - n_drop})
        n_in -= n_drop
    return out
all_mask = pd.Series(True, index=fl.index)
funnel = pd.DataFrame(funnel_rows(all_mask, "ALL"))
funnel_src = pd.DataFrame(sum([funnel_rows(src_col == s, s) for s in sorted(src_col.unique())], []))
assert funnel.iloc[-1]["features_out"] == len(final_feats), "funnel out at L5 != final_A count"
assert funnel.loc[funnel.layer == "L4", "features_out"].iloc[0] == len(pool_feats), "funnel out at L4 != pool"
src_pivot = funnel_src.pivot(index="source", columns="layer", values="features_out")[LAYERS]
src_pivot.insert(0, "candidates", src_col.value_counts().reindex(src_pivot.index))
drop_rules = (fl[fl["dropped_at"].notna()].groupby(["dropped_at", "drop_rule"]).size()
              .rename("n").reset_index().sort_values(["dropped_at", "n"], ascending=[True, False]))

tmp2 = os.path.join(_TMP, "03_funnel_summary.xlsx")
with pd.ExcelWriter(tmp2, engine="openpyxl") as w:
    _meta("E9_funnel_summary").to_excel(w, sheet_name="meta", index=False)
    funnel.to_excel(w, sheet_name="by_layer", index=False)
    src_pivot.to_excel(w, sheet_name="by_source")
    funnel_src.to_excel(w, sheet_name="by_layer_source", index=False)
    drop_rules.to_excel(w, sheet_name="drop_rules", index=False)
shutil.copyfile(tmp2, os.path.join(REPORTS, "03_funnel_summary.xlsx"))
print(f"Saved {REPORTS}/03_funnel_summary.xlsx")
display(funnel)
display(src_pivot.reset_index())

In [0]:
# ── MLflow: one run per module ──
import mlflow
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="E9_L5") as run:
    mlflow.set_tags({"module": "E9_L5", "git_commit": GIT_COMMIT})
    mlflow.log_params({"config_hash": common.config_hash(cfg), "funnel_config_hash": funnel_config_hash,
                       "c_grid": str(L5["c_grid"]), "n_subsamples": L5["n_subsamples"],
                       "subsample_frac": L5["subsample_frac"], "select_freq": str(L5["select_freq"]),
                       "select_freq_floor": L5["select_freq_floor"], "pvalue_max": PVALUE_MAX,
                       "vif_max": VIF_MAX, "seed": SEED, "binning_refit_per_fold": False})
    mlflow.log_metrics({"C_star": C_STAR, "best_C": float(BEST_C), "best_cv_auc": float(BEST_AUC),
                        "C_star_cv_auc": float(CSTAR_ROW["mean_auc"]), "select_freq_threshold": THR_USED,
                        "n_pool": len(pool_feats), "n_selected": len(sel_feats), "n_final": len(final_feats),
                        "max_p_value": float(logit_final.pvalues.drop("const").max()),
                        "max_vif": max(final_vif.values()),
                        "max_points_share": float(points_df["points_range_share"].max())})
    for p in [model_a_path, os.path.join(REPORTS, "03_L5.xlsx"), os.path.join(REPORTS, "03_funnel_summary.xlsx")]:
        mlflow.log_artifact(p)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run: {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance Criteria (E9 — L5) ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E9 — L5)"); print("=" * 60)
chk_fl = spark.table(FLOG_TBL).toPandas()
fin_coef = logit_final.params.drop("const")
fin_p = logit_final.pvalues.drop("const")
n_final = len(final_feats)
drop_ok = chk_fl["dropped_at"].astype(str).str.strip().isin(LAYERS)
final_ok = chk_fl["final_A"] == "Y"

checks = [
    (f"Final features in [{N_LO}, {N_HI}] (n={n_final})"
     + (" — below range, explained in report + open_questions.md" if few_explanation else ""),
     (N_LO <= n_final <= N_HI) or (n_final < N_LO and few_explanation is not None)),
    (f"All coefficient signs equal ({np.sign(fin_coef).value_counts().to_dict()})", np.sign(fin_coef).nunique() == 1),
    (f"All p < {PVALUE_MAX} (max={fin_p.max():.4g})", bool((fin_p < PVALUE_MAX).all())),
    (f"All VIF <= {VIF_MAX} (max={max(final_vif.values()):.2f})", max(final_vif.values()) <= VIF_MAX + 1e-9),
    ("Final Logit converged", bool(logit_final.mle_retvals["converged"])),
    (f"Every feature-log row has dropped_at or final_A=Y (missing={int((~drop_ok & ~final_ok).sum())})",
     bool((drop_ok ^ final_ok).all())),
    ("final_A=Y set == model_a.json features", set(chk_fl.loc[final_ok, "feature"]) == set(final_feats)),
    ("in_pool_BC=Y set == L4 pool", set(chk_fl.loc[chk_fl["in_pool_BC"] == "Y", "feature"]) == set(pool_feats)),
    (f"Feature-log rows unchanged ({len(chk_fl)} == {len(flog)})", len(chk_fl) == len(flog)),
    ("feature_log.csv exported", all(os.path.exists(p) for p in csv_paths)),
    ("model_a.json saved", os.path.exists(model_a_path)),
    ("03_L5.xlsx and 03_funnel_summary.xlsx saved",
     all(os.path.exists(os.path.join(REPORTS, x)) for x in ("03_L5.xlsx", "03_funnel_summary.xlsx"))),
]
all_pass = all(ok for _, ok in checks)
for desc, ok in checks:
    status = "\u2705 PASS" if ok else "\u274c FAIL"
    print(f"  {status} : {desc}")
print(f"  Points-share review flags (> {PTS_FLAG:.0%}): {flagged if flagged else 'none'}")

summary = {
    "module": "E9_L5", "run_time": datetime.now().isoformat(), "elapsed_s": round(time.time() - _t0, 1),
    "git_commit": GIT_COMMIT, "config_hash": common.config_hash(cfg), "funnel_config_hash": funnel_config_hash,
    "rows_dev_good_bad": len(dev_gb), "bads_dev": int(y.sum()),
    "features_in": len(pool_feats), "C_star": C_STAR, "best_C": float(BEST_C),
    "best_cv_auc": round(float(BEST_AUC), 4), "C_star_cv_auc": round(float(CSTAR_ROW["mean_auc"]), 4),
    "select_freq_threshold": THR_USED, "n_selected": len(sel_feats), "logit_dropped": list(l5_drop),
    "n_final": n_final, "final_features": final_feats,
    "max_p_value": float(fin_p.max()), "max_vif": round(max(final_vif.values()), 2),
    "points_review_flags": flagged, "mlflow_run_id": MLFLOW_RUN_ID,
    "binning_refit_per_fold": False, "pass_fail": "PASS" if all_pass else "FAIL",
}
with open(os.path.join(RUN_SUMS, "s03_L5.json"), "w") as fh:
    json.dump(summary, fh, indent=2, default=str)
print(f"\nRun summary: {RUN_SUMS}/s03_L5.json")

for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"
print(f"\n\u2705 All acceptance criteria passed. Model A features = {n_final}")